### The below cell updates the 1st Half scores from Ball Dont Lie API and puts them in the first_half_basics_table

In [ ]:
import os
import time
import requests
import psycopg2
from datetime import date, timedelta
import random

# ----------------------------
# CONFIG
# ----------------------------
API_KEY = os.getenv("BALLDONTLIE_API_KEY")
BASE_URL = "https://api.balldontlie.io/v1/games"

DB_CFG = dict(
    dbname="nba_data",
    user="postgres",
    password=os.getenv("NBA_DB_PASSWORD"),
    host="localhost",
    port=5432
)

PER_PAGE = 100
HEADERS = {"Authorization": API_KEY, "Accept": "application/json"}

THROTTLE_SEC = 1.5  # lighter for small backfills

def safe_get(session, params, max_retries=10):
    for attempt in range(1, max_retries + 1):
        time.sleep(THROTTLE_SEC)

        r = session.get(BASE_URL, headers=HEADERS, params=params, timeout=30)

        if r.status_code == 429:
            wait = min(10 * attempt, 120)
            print(f"  ⚠️ 429 Rate limit. Retry {attempt}/{max_retries} in {wait}s")
            time.sleep(wait)
            continue

        if r.status_code in (500, 502, 503, 504):
            wait = min(2 ** attempt, 30) + random.uniform(0, 1.5)
            print(f"  ⚠️ HTTP {r.status_code}. Retry {attempt}/{max_retries} in {wait:.1f}s")
            time.sleep(wait)
            continue

        r.raise_for_status()
        return r

    raise RuntimeError(f"Failed after {max_retries} retries for params={params}")

# ----------------------------
# DB CONNECT
# ----------------------------
print("Connecting to Postgres...")
conn = psycopg2.connect(**DB_CFG)
cur = conn.cursor()

# What’s the last date we have?
cur.execute("SELECT MAX(game_date) FROM first_half_basics;")
last_date = cur.fetchone()[0]

if last_date is None:
    raise RuntimeError("first_half_basics is empty. Use your full-season loader first.")

start_date = last_date + timedelta(days=1)
end_date = date.today() - timedelta(days=1)  # up through yesterday (safe)

if start_date > end_date:
    print(f"✅ No backfill needed. first_half_basics already up-to-date through {last_date}.")
    cur.close()
    conn.close()
    raise SystemExit

print(f"📆 Backfilling from {start_date} through {end_date}\n")

# ----------------------------
# LOAD MISSING DATES
# ----------------------------
session = requests.Session()

total_inserted = 0
total_attempted = 0

d = start_date
while d <= end_date:
    ds = d.strftime("%Y-%m-%d")
    print(f"=== Fetching games for {ds} ===")

    params = {
        "per_page": PER_PAGE,
        "dates[]": ds,
    }

    r = safe_get(session, params)
    payload = r.json()
    games = payload.get("data", []) or []

    print(f"  Games returned: {len(games)}")

    inserted_this_day = 0
    attempted_this_day = 0

    for g in games:
        home_q1 = g.get("home_q1")
        home_q2 = g.get("home_q2")
        vis_q1  = g.get("visitor_q1")
        vis_q2  = g.get("visitor_q2")

        # If the API didn't provide quarter scoring, skip
        if None in (home_q1, home_q2, vis_q1, vis_q2):
            continue

        try:
            home_1h = int(home_q1) + int(home_q2)
            away_1h = int(vis_q1) + int(vis_q2)
        except (ValueError, TypeError):
            continue

        home = g["home_team"]["abbreviation"]
        away = g["visitor_team"]["abbreviation"]
        game_date = (g.get("date") or "")[:10]
        game_id = g["id"]

        team_rows = [
            (game_id, game_date, home, away, "H", home_1h, away_1h),
            (game_id, game_date, away, home, "A", away_1h, home_1h),
        ]

        for game_id, gd, team, opp, site, pts_for, pts_against in team_rows:
            margin = pts_for - pts_against
            result = "W" if margin > 0 else "L" if margin < 0 else "T"

            cur.execute("""
                INSERT INTO first_half_basics (
                    game_id, game_date, team, opponent, site,
                    first_half_result, pts_scored_1, pts_allowed, margin
                )
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
                ON CONFLICT (game_id, team) DO NOTHING;
            """, (game_id, gd, team, opp, site, result, pts_for, pts_against, margin))

            attempted_this_day += 1
            if cur.rowcount == 1:
                inserted_this_day += 1

    conn.commit()

    print(f"  ✅ {ds} committed. Inserted {inserted_this_day}/{attempted_this_day} attempted rows.\n")

    total_inserted += inserted_this_day
    total_attempted += attempted_this_day

    d += timedelta(days=1)

cur.close()
conn.close()

print(f"🏁 Backfill complete. Total inserted: {total_inserted} (attempted {total_attempted}).")


Connecting to Postgres...
📆 Backfilling from 2026-03-24 through 2026-03-24

=== Fetching games for 2026-03-24 ===
  Games returned: 4
  ✅ 2026-03-24 committed. Inserted 8/8 attempted rows.

🏁 Backfill complete. Total inserted: 8 (attempted 8).


## The below cell derives L5 rolling stats and populates fh_rolling_five_table

In [ ]:
import os
# ==================================================
# FH ROLLING 5 – FULL PIPELINE (SQL → PANDAS → SQL) + wpct
# wpct = last-5 "win%" where win=1, loss=0, tie=0.5
# ==================================================

import pandas as pd
from sqlalchemy import create_engine, text

# --------------------------------------------------
# DB CONNECTION
# --------------------------------------------------
engine = create_engine(
    os.getenv("NBA_DATABASE_URL")
)

print("🔌 Connected to Postgres")

# --------------------------------------------------
# LOAD SOURCE DATA
# --------------------------------------------------
query = """
SELECT
    game_date,
    team,
    site AS split,
    pts_scored_1,
    pts_allowed,
    margin
FROM first_half_basics
ORDER BY team, split, game_date;
"""

df = pd.read_sql(query, engine)
df["game_date"] = pd.to_datetime(df["game_date"])

print(f"📥 Loaded {len(df)} rows from first_half_basics")

# --------------------------------------------------
# NBA SEASON LOGIC
# --------------------------------------------------
df["season"] = df["game_date"].dt.year
df.loc[df["game_date"].dt.month < 10, "season"] -= 1
print("🗓️ Season column created")

# --------------------------------------------------
# SORT FOR ROLLING WINDOWS
# --------------------------------------------------
df = df.sort_values(["team", "split", "season", "game_date"])

group_cols = ["team", "split", "season"]

# --------------------------------------------------
# ROLLING 5-GAME CALCULATIONS
# --------------------------------------------------
df["rolling_5_pts"] = (
    df.groupby(group_cols)["pts_scored_1"]
      .rolling(5)
      .mean()
      .reset_index(level=group_cols, drop=True)
)

df["rolling_5_pts_allowed"] = (
    df.groupby(group_cols)["pts_allowed"]
      .rolling(5)
      .mean()
      .reset_index(level=group_cols, drop=True)
)

df["rolling_5_margin"] = (
    df.groupby(group_cols)["margin"]
      .rolling(5)
      .mean()
      .reset_index(level=group_cols, drop=True)
)

print("📊 Rolling 5-game metrics computed")

# --------------------------------------------------
# GAME COUNT (MATCH SQL COUNT OVER WINDOW)
# --------------------------------------------------
df["game_count"] = df.groupby(group_cols).cumcount() + 1

# --------------------------------------------------
# wpct (last 5) with ties = 0.5
# --------------------------------------------------
df["win_value"] = 0.0
df.loc[df["margin"] > 0, "win_value"] = 1.0
df.loc[df["margin"] == 0, "win_value"] = 0.5

df["wpct"] = (
    df.groupby(group_cols)["win_value"]
      .rolling(5)
      .mean()
      .reset_index(level=group_cols, drop=True)
)

print("🏆 Rolling 5-game wpct computed (ties=0.5)")

# --------------------------------------------------
# NULL OUT ROWS WITHOUT FULL WINDOW (5 games)
# --------------------------------------------------
mask_incomplete = df["game_count"] < 5
df.loc[mask_incomplete, ["rolling_5_pts", "rolling_5_pts_allowed", "rolling_5_margin", "wpct"]] = None

# --------------------------------------------------
# FINAL OUTPUT TABLE (MATCH SQL OUTPUT + wpct)
# --------------------------------------------------
final_df = (
    df[[
        "game_date",
        "team",
        "split",
        "rolling_5_pts",
        "rolling_5_pts_allowed",
        "rolling_5_margin",
        "wpct"
    ]]
    .rename(columns={"game_date": "date"})
    .sort_values(["team", "split", "date"])
)

# Round display nicely (wpct looks better with 3 decimals)
final_df["rolling_5_pts"] = final_df["rolling_5_pts"].round(2)
final_df["rolling_5_pts_allowed"] = final_df["rolling_5_pts_allowed"].round(2)
final_df["rolling_5_margin"] = final_df["rolling_5_margin"].round(2)
final_df["wpct"] = final_df["wpct"].round(3)

print(f"📦 Final dataset rows: {len(final_df)}")

# --------------------------------------------------
# WRITE TO POSTGRES (SQLAlchemy 2.x safe)
# --------------------------------------------------
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS fh_rolling_five_raw;"))

final_df.to_sql(
    "fh_rolling_five_raw",
    engine,
    index=False,
    if_exists="replace"
)

print("✅ fh_rolling_five_raw created successfully (includes wpct)")
print(final_df.tail(10))

🔌 Connected to Postgres
📥 Loaded 7438 rows from first_half_basics
🗓️ Season column created
📊 Rolling 5-game metrics computed
🏆 Rolling 5-game wpct computed (ties=0.5)
📦 Final dataset rows: 7438
✅ fh_rolling_five_raw created successfully (includes wpct)
           date team split  rolling_5_pts  rolling_5_pts_allowed  \
7428 2026-02-19  WAS     H           51.4                   63.6   
7429 2026-02-20  WAS     H           53.2                   61.8   
7430 2026-02-22  WAS     H           53.8                   64.4   
7431 2026-02-28  WAS     H           57.6                   62.2   
7432 2026-03-02  WAS     H           57.4                   59.4   
7433 2026-03-05  WAS     H           55.4                   63.4   
7434 2026-03-16  WAS     H           55.4                   62.6   
7435 2026-03-17  WAS     H           55.6                   63.0   
7436 2026-03-19  WAS     H           53.2                   62.8   
7437 2026-03-21  WAS     H           55.8                   64.6   

### The below cell uses current machine learning logic from ML notebook to handicap the days card

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
from datetime import datetime, date
from zoneinfo import ZoneInfo
from sqlalchemy import create_engine, text

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

BALLDONTLIE_API_KEY = os.getenv("BALLDONTLIE_API_KEY")
DB_URL = os.getenv("NBA_DATABASE_URL")

PHX = ZoneInfo("America/Phoenix")
today_str = datetime.now(PHX).strftime("%Y-%m-%d")

CSV_PATH = rf"C:\Users\padre\MLB_Code_2024\nba_1st_half\daily_inference_{today_str}.csv"

CORE_MIN, CORE_MAX = 0.61, 0.63
LEAN_MIN, LEAN_MAX = 0.59, 0.61

engine = create_engine(DB_URL)
print(f"📅 Phoenix date: {today_str}")
print("🔌 Connected to Postgres")

# ----------------------------
# Helper: Probability -> American Moneyline
# ----------------------------
def prob_to_moneyline(p):
    if pd.isna(p):
        return np.nan
    # guard against 0/1 exactly
    p = float(p)
    eps = 1e-9
    p = min(max(p, eps), 1 - eps)

    if p >= 0.5:
        return int(round(-100 * (p / (1 - p))))
    else:
        return int(round(100 * ((1 - p) / p)))

# ----------------------------
# 1) TRAIN CALIBRATED LOGISTIC
# ----------------------------
train_df = pd.read_sql(
    text("SELECT * FROM fh_daily_ml_2023_onward_clean ORDER BY game_date"),
    engine
)

train_df = train_df[train_df["home_win"].isin([0.0, 1.0])].copy()
train_df["home_win"] = train_df["home_win"].astype(int)

feature_cols = [
    "home_r5_pts", "home_r5_allowed", "home_r5_margin", "home_wpct",
    "away_r5_pts", "away_r5_allowed", "away_r5_margin", "away_wpct",
    "home_plus_minus", "pct_margin"
]

X_all = train_df[feature_cols]
y_all = train_df["home_win"]

train_mask = train_df["game_date"] < date(2025, 1, 1)
X_train, y_train = X_all[train_mask], y_all[train_mask]

base_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000))
])

cal_model = CalibratedClassifierCV(estimator=base_pipe, method="sigmoid", cv=5)

print(f"🧠 Training calibrated logistic on {len(X_train)} rows...")
cal_model.fit(X_train, y_train)
print("✅ Model trained")

# ----------------------------
# 2) TODAY'S SCHEDULE
# ----------------------------
print("🌐 Fetching today's NBA games...")
url = "https://api.balldontlie.io/v1/games"
params = {"dates[]": today_str, "per_page": 100}
headers = {"Authorization": BALLDONTLIE_API_KEY}

resp = requests.get(url, params=params, headers=headers, timeout=30)
resp.raise_for_status()
games = resp.json().get("data", [])
print(f"✅ Games found: {len(games)}")

if not games:
    print("No NBA games today.")
    raise SystemExit

schedule_rows = []
for g in games:
    home = g.get("home_team", {}) or {}
    away = g.get("visitor_team", {}) or {}
    schedule_rows.append({
        "game_id": g.get("id"),
        "status": g.get("status"),
        "home_team": home.get("abbreviation"),
        "away_team": away.get("abbreviation"),
        "start_utc": g.get("datetime"),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values("game_id")

# ----------------------------
# 3) LATEST SPLIT ROLLING-5 (incl wpct)
# ----------------------------
print("\n📊 Pulling latest split rolling-5 from fh_rolling_five_raw (incl wpct)...")

latest_sql = """
WITH latest AS (
    SELECT DISTINCT ON (team, split)
        team, split, date,
        rolling_5_pts, rolling_5_pts_allowed, rolling_5_margin, wpct
    FROM fh_rolling_five_raw
    WHERE rolling_5_margin IS NOT NULL
    ORDER BY team, split, date DESC
)
SELECT * FROM latest;
"""

roll_df = pd.read_sql(text(latest_sql), engine)
roll_df["date"] = pd.to_datetime(roll_df["date"])

home_roll = (
    roll_df[roll_df["split"] == "H"]
    .rename(columns={
        "date": "home_roll_date",
        "rolling_5_pts": "home_r5_pts",
        "rolling_5_pts_allowed": "home_r5_allowed",
        "rolling_5_margin": "home_r5_margin",
        "wpct": "home_wpct"
    })
    .drop(columns=["split"])
)

away_roll = (
    roll_df[roll_df["split"] == "A"]
    .rename(columns={
        "date": "away_roll_date",
        "rolling_5_pts": "away_r5_pts",
        "rolling_5_pts_allowed": "away_r5_allowed",
        "rolling_5_margin": "away_r5_margin",
        "wpct": "away_wpct"
    })
    .drop(columns=["split"])
)

out = (
    schedule_df
    .merge(home_roll, left_on="home_team", right_on="team", how="left")
    .drop(columns=["team"])
    .merge(away_roll, left_on="away_team", right_on="team", how="left")
    .drop(columns=["team"])
)

# Pre-game edges (same definition as training)
out["home_plus_minus"] = (out["home_r5_margin"] - out["away_r5_margin"]).round(2)
denom = out["home_r5_margin"].abs() + out["away_r5_margin"].abs()
out["pct_margin"] = np.where(denom.notna() & (denom != 0), out["home_plus_minus"] / denom, np.nan)
out["pct_margin"] = pd.Series(out["pct_margin"]).round(3)

# Ensure these exist on all rows (prevents KeyError)
out["home_win_proba"] = np.nan
out["model_moneyline"] = np.nan
out["signal"] = "NO DATA"

# Score only rows with complete features
out["has_features"] = out[feature_cols].notna().all(axis=1)
scored = out[out["has_features"]].copy()

if len(scored) > 0:
    scored["home_win_proba"] = cal_model.predict_proba(scored[feature_cols])[:, 1]
    scored["home_win_proba"] = scored["home_win_proba"].round(3)

    def flag(p):
        if CORE_MIN <= p <= CORE_MAX:
            return "CORE PLAY"
        if LEAN_MIN <= p < LEAN_MAX:
            return "LEAN"
        return "PASS"

    scored["signal"] = scored["home_win_proba"].apply(flag)
    scored["model_moneyline"] = scored["home_win_proba"].apply(prob_to_moneyline)

    # write back into out
    out.loc[scored.index, "home_win_proba"] = scored["home_win_proba"]
    out.loc[scored.index, "model_moneyline"] = scored["model_moneyline"]
    out.loc[scored.index, "signal"] = scored["signal"]

# Final columns
final_cols = [
    "home_team", "away_team", "status", "start_utc",
    "home_r5_pts", "home_r5_allowed", "home_r5_margin", "home_wpct", "home_roll_date",
    "away_r5_pts", "away_r5_allowed", "away_r5_margin", "away_wpct", "away_roll_date",
    "home_plus_minus", "pct_margin",
    "home_win_proba", "signal", "model_moneyline", 
]

final_df = out[final_cols].sort_values(["signal", "home_win_proba"], ascending=[True, False])

print("\n🏀 TODAY — CALIBRATED PROBABILITIES + MODEL MONEYLINE + SIGNALS")
print(final_df.to_string(index=False))

final_df.to_csv(CSV_PATH, index=False)
print(f"\n📁 CSV written to: {CSV_PATH}")



📅 Phoenix date: 2026-03-25
🔌 Connected to Postgres
🧠 Training calibrated logistic on 1671 rows...
✅ Model trained
🌐 Fetching today's NBA games...
✅ Games found: 12

📊 Pulling latest split rolling-5 from fh_rolling_five_raw (incl wpct)...

🏀 TODAY — CALIBRATED PROBABILITIES + MODEL MONEYLINE + SIGNALS
home_team away_team               status                start_utc  home_r5_pts  home_r5_allowed  home_r5_margin  home_wpct home_roll_date  away_r5_pts  away_r5_allowed  away_r5_margin  away_wpct away_roll_date  home_plus_minus  pct_margin  home_win_proba    signal  model_moneyline
      MIN       HOU 2026-03-26T01:30:00Z 2026-03-26T01:30:00.000Z         60.0             60.6            -0.6        0.6     2026-03-20         52.8             57.8            -5.0        0.4     2026-03-23              4.4       0.786           0.614 CORE PLAY           -159.0
      GSW       BKN 2026-03-26T02:00:00Z 2026-03-26T02:00:00.000Z         55.6             60.0            -4.4        0.4     2026-03

In [ ]:
import numpy as np

# ------------------------------------------------------------
# HOME WIN PROBABILITY FROM SIMPLE CALIBRATION EQUATION
# y = 0.232x + 0.5333
# where x = (home_wpct - away_wpct)
# ------------------------------------------------------------

SLOPE = 0.232
INTERCEPT = 0.5333

# compute wpct diff + projection
final_df["wpct_diff"] = final_df["home_wpct"] - final_df["away_wpct"]
final_df["home_win_proba_eq"] = INTERCEPT + SLOPE * final_df["wpct_diff"]

# clip to [0,1] so we never get impossible probabilities
final_df["home_win_proba_eq"] = final_df["home_win_proba_eq"].clip(0, 1)

# (optional) quick view of what you asked for
cols = ["home_team", "away_team", "home_wpct", "away_wpct", "wpct_diff", "home_win_proba_eq"]
display(final_df[cols].sort_values("home_win_proba_eq", ascending=False))


,home_team,away_team,home_wpct,away_wpct,wpct_diff,home_win_proba_eq
6,UTA,WAS,0.8,0.0,0.8,0.7189
10,POR,MIL,0.8,0.2,0.6,0.6725
4,CLE,MIA,0.6,0.0,0.6,0.6725
0,DET,ATL,1.0,0.4,0.6,0.6725
8,DEN,DAL,0.8,0.4,0.4,0.6261
9,GSW,BKN,0.4,0.2,0.2,0.5797
7,MIN,HOU,0.6,0.4,0.2,0.5797
11,LAC,TOR,0.6,0.4,0.2,0.5797
2,PHI,CHI,0.4,0.4,0.0,0.5333
1,IND,LAL,0.0,0.2,-0.2,0.4869


## The below code generates an updated list of 1st half records

### Step 1 -Python: Pull + Aggregate Everything (One Pass)

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from datetime import date, timedelta

# =========================
# DB CONNECTION
# =========================
DB_URL = os.getenv("NBA_DATABASE_URL")
engine = create_engine(DB_URL)

SEASON_START = "2025-10-01"  # 2025-26 season anchor

# =========================
# LOAD 2025-26 SEASON ONLY
# =========================
sql = f"""
SELECT
    game_id,
    game_date::date AS game_date,
    team,
    site,
    first_half_result,
    pts_scored_1::numeric AS pts_scored_1,
    pts_allowed::numeric  AS pts_allowed,
    margin::numeric       AS margin
FROM first_half_basics
WHERE game_date >= '{SEASON_START}'
ORDER BY game_date;
"""

df = pd.read_sql(sql, engine)
print(f"Loaded {len(df):,} first-half rows for 2025-26 season (>= {SEASON_START})")

# =========================
# SUMMARY HELPERS
# =========================
def _base_agg(dfx: pd.DataFrame) -> dict:
    games = int(len(dfx))
    w = int((dfx["first_half_result"] == "W").sum())
    l = int((dfx["first_half_result"] == "L").sum())
    t = int((dfx["first_half_result"] == "T").sum())

    pts_scored = float(dfx["pts_scored_1"].sum()) if games else 0.0
    pts_allowed = float(dfx["pts_allowed"].sum()) if games else 0.0

    avg_pts_scored = float(dfx["pts_scored_1"].mean()) if games else 0.0
    avg_pts_allowed = float(dfx["pts_allowed"].mean()) if games else 0.0
    avg_margin = float(dfx["margin"].mean()) if games else 0.0

    win_pct = round((w / games) * 100, 1) if games else 0.0

    return {
        "games": games,
        "w": w,
        "l": l,
        "t": t,
        "win_pct": win_pct,
        "pts_scored": round(pts_scored, 2),
        "pts_allowed": round(pts_allowed, 2),
        "avg_pts_scored": round(avg_pts_scored, 2),
        "avg_pts_allowed": round(avg_pts_allowed, 2),
        "avg_margin": round(avg_margin, 2),
    }

def summarize_teams(dfx: pd.DataFrame, split_label: str) -> pd.DataFrame:
    rows = []
    for team, g in dfx.groupby("team"):
        r = _base_agg(g)
        r.update({"level": "TEAM", "entity": team, "split": split_label})
        rows.append(r)
    return pd.DataFrame(rows)

def summarize_league(dfx: pd.DataFrame, split_label: str) -> pd.DataFrame:
    r = _base_agg(dfx)
    r.update({"level": "LEAGUE", "entity": "NBA", "split": split_label})
    return pd.DataFrame([r])

# =========================
# BUILD SPLITS (TEAMS + LEAGUE)
# =========================
today = date.today()

def add_split(dfx: pd.DataFrame, split_label: str, out_list: list):
    out_list.append(summarize_teams(dfx, split_label))
    out_list.append(summarize_league(dfx, split_label))

parts = []
add_split(df, "YTD", parts)
add_split(df[df.site == "H"], "HOME", parts)
add_split(df[df.site == "A"], "AWAY", parts)
add_split(df[df.game_date >= today - timedelta(days=7)], "L7", parts)
add_split(df[df.game_date >= today - timedelta(days=14)], "L14", parts)
add_split(df[df.game_date >= today - timedelta(days=30)], "L30", parts)

ytd_df = pd.concat(parts, ignore_index=True)

# =========================
# SORT (LEAGUE FIRST, THEN TEAMS)
# =========================
split_order = ["YTD", "HOME", "AWAY", "L30", "L14", "L7"]
ytd_df["split_rank"] = ytd_df["split"].map({s: i for i, s in enumerate(split_order)})
ytd_df["level_rank"] = ytd_df["level"].map({"LEAGUE": 0, "TEAM": 1})

ytd_df = (
    ytd_df
    .sort_values(["level_rank", "entity", "split_rank"])
    .drop(columns=["split_rank", "level_rank"])
)

# =========================
# SAVE SINGLE CSV
# =========================
output_file = "nba_2025_26_1h_team_plus_league_splits.csv"
ytd_df.to_csv(output_file, index=False)

print(f"✅ Saved {output_file}")
print(ytd_df.head(20))


Loaded 2,160 first-half rows for 2025-26 season (>= 2025-10-01)
✅ Saved nba_2025_26_1h_team_plus_league_splits.csv
     games     w     l   t  win_pct  pts_scored  pts_allowed  avg_pts_scored  \
30    2160  1054  1054  52     48.8    125425.0     125425.0           58.07   
61    1080   560   494  26     51.9     63139.0      62286.0           58.46   
92    1080   494   560  26     45.7     62286.0      63139.0           57.67   
185    448   217   217  14     48.4     25972.0      25972.0           57.97   
154    210   102   102   6     48.6     12357.0      12357.0           58.84   
123    104    50    50   4     48.1      6191.0       6191.0           59.53   
0       72    38    32   2     52.8      4334.0       4253.0           60.19   
31      37    20    16   1     54.1      2211.0       2134.0           59.76   
62      35    18    16   1     51.4      2123.0       2119.0           60.66   
155     13    10     3   0     76.9       846.0        731.0           65.08   
124  

In [ ]:
# ==========================================
# TOP / BOTTOM 5: HOME vs AWAY WIN %
# ==========================================

# Keep only TEAM rows and HOME / AWAY splits
ha = (
    ytd_df[
        (ytd_df["level"] == "TEAM") &
        (ytd_df["split"].isin(["HOME", "AWAY"]))
    ]
    .pivot(index="entity", columns="split", values="win_pct")
    .reset_index()
)

# Drop teams missing either split
ha = ha.dropna(subset=["HOME", "AWAY"])

# Calculate differential
ha["home_minus_away_win_pct"] = (ha["HOME"] - ha["AWAY"]).round(1)

# Rename for clarity
ha = ha.rename(columns={"entity": "team"})

# Sort
ha_sorted = ha.sort_values("home_minus_away_win_pct", ascending=False)

# Top 5 (best home edge)
top_5_home_edge = ha_sorted.head(5)

# Bottom 5 (worst home edge / better away)
bottom_5_home_edge = ha_sorted.tail(5).sort_values(
    "home_minus_away_win_pct"
)

print("\n🔥 TOP 5 — Biggest HOME Advantage (1H Win %)")
print(top_5_home_edge)

print("\n❄️ BOTTOM 5 — Worst HOME Advantage / Better AWAY (1H Win %)")
print(bottom_5_home_edge)



🔥 TOP 5 — Biggest HOME Advantage (1H Win %)
split team  AWAY  HOME  home_minus_away_win_pct
7      DEN  43.6  70.6                     27.0
28     UTA  25.7  51.4                     25.7
12     LAC  43.2  65.7                     22.5
15     MIA  48.6  67.6                     19.0
13     LAL  35.1  51.4                     16.3

❄️ BOTTOM 5 — Worst HOME Advantage / Better AWAY (1H Win %)
split team  AWAY  HOME  home_minus_away_win_pct
21     ORL  54.3  40.5                    -13.8
20     OKC  77.8  66.7                    -11.1
23     PHX  54.3  47.4                     -6.9
11     IND  35.1  28.6                     -6.5
2      BOS  69.4  65.7                     -3.7


In [ ]:
ytd_df


,games,w,l,t,win_pct,pts_scored,pts_allowed,avg_pts_scored,avg_pts_allowed,avg_margin,level,entity,split
30,2160,1054,1054,52,48.8,125425.0,125425.0,58.07,58.07,0.00,LEAGUE,NBA,YTD
61,1080,560,494,26,51.9,63139.0,62286.0,58.46,57.67,0.79,LEAGUE,NBA,HOME
92,1080,494,560,26,45.7,62286.0,63139.0,57.67,58.46,-0.79,LEAGUE,NBA,AWAY
185,448,217,217,14,48.4,25972.0,25972.0,57.97,57.97,0.00,LEAGUE,NBA,L30
154,210,102,102,6,48.6,12357.0,12357.0,58.84,58.84,0.00,LEAGUE,NBA,L14
...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,37,12,25,0,32.4,2114.0,2282.0,57.14,61.68,-4.54,TEAM,WAS,HOME
91,34,7,27,0,20.6,1884.0,2207.0,55.41,64.91,-9.50,TEAM,WAS,AWAY
184,15,1,14,0,6.7,823.0,975.0,54.87,65.00,-10.13,TEAM,WAS,L30
153,7,0,7,0,0.0,379.0,451.0,54.14,64.43,-10.29,TEAM,WAS,L14


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

SEASON_START = "2025-10-01"

# Make sure this points to your nba_data DB
PG_URL = os.getenv(
    "NBA_PG_URL",
    os.getenv("NBA_DATABASE_URL")
)
engine = create_engine(PG_URL)

# ----------------------------
# 1) Pull YTD 1H W-L-(T) from first_half_basics
# ----------------------------
sql = text("""
    SELECT
        team,
        SUM(CASE WHEN first_half_result = 'W' THEN 1 ELSE 0 END) AS w,
        SUM(CASE WHEN first_half_result = 'L' THEN 1 ELSE 0 END) AS l,
        SUM(CASE WHEN first_half_result = 'T' THEN 1 ELSE 0 END) AS t
    FROM public.first_half_basics
    WHERE game_date >= :season_start
    GROUP BY team
""")

team_ytd = pd.read_sql(sql, engine, params={"season_start": SEASON_START})

# Ensure ints + record string
team_ytd[["w","l","t"]] = team_ytd[["w","l","t"]].fillna(0).astype(int)
team_ytd["ytd_1h_record"] = team_ytd.apply(lambda r: f"{r.w}-{r.l}-{r.t}", axis=1)
team_ytd = team_ytd[["team", "ytd_1h_record"]]

# ----------------------------
# 2) Merge into today's matchups (final_df)
# ----------------------------
matchups_ytd = (
    final_df[["home_team", "away_team"]].copy()
    .merge(team_ytd, left_on="home_team", right_on="team", how="left")
    .rename(columns={"ytd_1h_record": "home_ytd_1h_record"})
    .drop(columns=["team"])
    .merge(team_ytd, left_on="away_team", right_on="team", how="left")
    .rename(columns={"ytd_1h_record": "away_ytd_1h_record"})
    .drop(columns=["team"])
)

# ----------------------------
# 3) Add model cols if you have them
# ----------------------------
model_cols = [c for c in ["home_win_proba", "signal", "model_moneyline"] if c in final_df.columns]
if model_cols:
    matchups_ytd = matchups_ytd.merge(
        final_df[["home_team", "away_team"] + model_cols],
        on=["home_team", "away_team"],
        how="left"
    )

keep_cols = ["home_team","home_ytd_1h_record","away_team","away_ytd_1h_record"] + model_cols
display(matchups_ytd[keep_cols])



,home_team,home_ytd_1h_record,away_team,away_ytd_1h_record,home_win_proba,signal,model_moneyline
0,MIN,37-32-3,HOU,39-30-2,0.614,CORE PLAY,-159.0
1,GSW,38-34-0,BKN,21-50-1,0.597,LEAN,-148.0
2,UTA,28-43-1,WAS,19-52-0,0.713,PASS,-248.0
3,POR,33-38-2,MIL,29-42-0,0.685,PASS,-217.0
4,DET,51-19-1,ATL,38-32-2,0.668,PASS,-201.0
5,DEN,41-28-4,DAL,25-46-1,0.650,PASS,-186.0
6,CLE,36-35-1,MIA,42-28-2,0.631,PASS,-171.0
7,PHI,41-29-2,CHI,24-45-2,0.549,PASS,-122.0
8,LAC,39-31-2,TOR,38-33-0,0.541,PASS,-118.0
9,IND,23-48-1,LAL,31-37-4,0.448,PASS,123.0


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

SEASON_START = "2025-10-01"

PG_URL = os.getenv(
    "NBA_PG_URL",
    os.getenv("NBA_DATABASE_URL")
)
engine = create_engine(PG_URL)

# ----------------------------
# 1) Pull TEAM x SITE split records
# ----------------------------
sql = text("""
    SELECT
        team,
        site,
        SUM(CASE WHEN first_half_result = 'W' THEN 1 ELSE 0 END) AS w,
        SUM(CASE WHEN first_half_result = 'L' THEN 1 ELSE 0 END) AS l,
        SUM(CASE WHEN first_half_result = 'T' THEN 1 ELSE 0 END) AS t
    FROM public.first_half_basics
    WHERE game_date >= :season_start
    GROUP BY team, site
""")

split_df = pd.read_sql(sql, engine, params={"season_start": SEASON_START})
split_df[["w","l","t"]] = split_df[["w","l","t"]].fillna(0).astype(int)
split_df["record"] = split_df.apply(lambda r: f"{r.w}-{r.l}-{r.t}", axis=1)

# pivot to wide: one row per team, columns for H/A records
team_site = (
    split_df.pivot(index="team", columns="site", values="record")
    .reset_index()
    .rename(columns={"H": "ytd_1h_home_record", "A": "ytd_1h_away_record"})
)

# ----------------------------
# 2) Merge into today's matchups
# ----------------------------
matchups = final_df[["home_team", "away_team"]].copy()

# home team uses its HOME split
matchups = matchups.merge(
    team_site[["team", "ytd_1h_home_record"]],
    left_on="home_team",
    right_on="team",
    how="left"
).drop(columns=["team"]).rename(columns={"ytd_1h_home_record": "home_ytd_1h_record"})

# away team uses its AWAY split
matchups = matchups.merge(
    team_site[["team", "ytd_1h_away_record"]],
    left_on="away_team",
    right_on="team",
    how="left"
).drop(columns=["team"]).rename(columns={"ytd_1h_away_record": "away_ytd_1h_record"})

# ----------------------------
# 3) Add model cols if present
# ----------------------------
model_cols = [c for c in ["home_win_proba", "signal", "model_moneyline"] if c in final_df.columns]
if model_cols:
    matchups = matchups.merge(
        final_df[["home_team", "away_team"] + model_cols],
        on=["home_team", "away_team"],
        how="left"
    )

keep_cols = ["home_team","home_ytd_1h_record","away_team","away_ytd_1h_record"] + model_cols
display(matchups[keep_cols])



,home_team,home_ytd_1h_record,away_team,away_ytd_1h_record,home_win_proba,signal,model_moneyline
0,MIN,19-17-1,HOU,19-15-2,0.614,CORE PLAY,-159.0
1,GSW,19-15-0,BKN,8-29-0,0.597,LEAN,-148.0
2,UTA,19-18-0,WAS,7-27-0,0.713,PASS,-248.0
3,POR,17-17-1,MIL,15-21-0,0.685,PASS,-217.0
4,DET,26-9-1,ATL,18-16-1,0.668,PASS,-201.0
5,DEN,24-10-0,DAL,12-23-0,0.650,PASS,-186.0
6,CLE,18-18-0,MIA,17-17-1,0.631,PASS,-171.0
7,PHI,22-14-1,CHI,10-22-1,0.549,PASS,-122.0
8,LAC,23-11-1,TOR,18-18-0,0.541,PASS,-118.0
9,IND,10-24-1,LAL,13-21-3,0.448,PASS,123.0


In [ ]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

SEASON_START = "2025-10-01"

PG_URL = os.getenv(
    "NBA_PG_URL",
    os.getenv("NBA_DATABASE_URL")
)
engine = create_engine(PG_URL)

# Pull team x site counts
sql = text("""
    SELECT
        team,
        site,
        SUM(CASE WHEN first_half_result = 'W' THEN 1 ELSE 0 END) AS w,
        SUM(CASE WHEN first_half_result = 'L' THEN 1 ELSE 0 END) AS l,
        SUM(CASE WHEN first_half_result = 'T' THEN 1 ELSE 0 END) AS t,
        COUNT(*) AS games
    FROM public.first_half_basics
    WHERE game_date >= :season_start
    GROUP BY team, site
""")

ytd_site = pd.read_sql(sql, engine, params={"season_start": SEASON_START})
ytd_site[["w","l","t","games"]] = ytd_site[["w","l","t","games"]].fillna(0).astype(int)

# Record string + wpct (ties count as 0.5 here; change if you want ties ignored)
ytd_site["record"] = ytd_site.apply(lambda r: f"{r.w}-{r.l}-{r.t}", axis=1)
ytd_site["wpct"] = np.where(
    ytd_site["games"] > 0,
    (ytd_site["w"] + 0.5 * ytd_site["t"]) / ytd_site["games"],
    np.nan
)

# Wide: one row per team with HOME + AWAY record/wpct
ytd_wide = (
    ytd_site.pivot(index="team", columns="site", values=["record", "wpct"])
    .reset_index()
)

# Flatten columns
ytd_wide.columns = [
    "team" if c[0] == "team" else f"ytd_{c[1].lower()}_{c[0]}"
    for c in ytd_wide.columns
]

# Expect these:
# ytd_h_record, ytd_a_record, ytd_h_wpct, ytd_a_wpct
# If any missing, it's because a team has no games at that site yet.
ytd_wide.head()




,team,ytd_a_record,ytd_h_record,ytd_a_wpct,ytd_h_wpct
0,ATL,18-16-1,20-16-1,0.528571,0.554054
1,BKN,8-29-0,13-21-1,0.216216,0.385714
2,BOS,25-9-2,23-10-2,0.722222,0.685714
3,CHA,18-17-2,21-14-0,0.513514,0.6
4,CHI,10-22-1,14-23-1,0.318182,0.381579


In [ ]:
import numpy as np

# ------------------------------------------------------------
# Your existing calibration equation
# y = 0.232x + 0.5333
# x = (home_metric - away_metric)
# ------------------------------------------------------------
SLOPE = 0.232
INTERCEPT = 0.5333

out = final_df.copy()

# --- 1) Merge in YTD site splits (home team uses H, away team uses A) ---
out = out.merge(
    ytd_wide[["team", "ytd_h_record", "ytd_h_wpct"]],
    left_on="home_team",
    right_on="team",
    how="left"
).drop(columns=["team"]).rename(columns={
    "ytd_h_record": "home_ytd_home_record",
    "ytd_h_wpct": "home_ytd_home_wpct"
})

out = out.merge(
    ytd_wide[["team", "ytd_a_record", "ytd_a_wpct"]],
    left_on="away_team",
    right_on="team",
    how="left"
).drop(columns=["team"]).rename(columns={
    "ytd_a_record": "away_ytd_away_record",
    "ytd_a_wpct": "away_ytd_away_wpct"
})

# --- 2) L5 H/A probability (already based on home_wpct/away_wpct in final_df) ---
# If you already created home_win_proba_eq earlier, we'll reuse it.
if "home_win_proba_eq" in out.columns:
    out["l5_ha_proba"] = out["home_win_proba_eq"]
else:
    # fallback: compute it here if needed
    out["wpct_diff_l5"] = out["home_wpct"] - out["away_wpct"]
    out["l5_ha_proba"] = (INTERCEPT + SLOPE * out["wpct_diff_l5"]).clip(0, 1)

# --- 3) YTD H/A split probability (HOME team's HOME wpct vs AWAY team's AWAY wpct) ---
out["wpct_diff_ytd"] = out["home_ytd_home_wpct"] - out["away_ytd_away_wpct"]
out["ytd_ha_proba"] = (INTERCEPT + SLOPE * out["wpct_diff_ytd"]).clip(0, 1)

# --- 4) Model probability (already in final_df as home_win_proba) ---
out["model_proba"] = out["home_win_proba"] if "home_win_proba" in out.columns else np.nan

# ----------------------------
# Final columns (single output)
# ----------------------------
final_cols = [c for c in [
    "home_team",
    "home_ytd_home_record",
    "away_team",
    "away_ytd_away_record",

    # show the wpcts used
    "home_ytd_home_wpct",
    "away_ytd_away_wpct",

    # the 3 probabilities
    "model_proba",
    "l5_ha_proba",
    "ytd_ha_proba",

    # keep your model outputs if present
    "signal",
    "model_moneyline",

    # optional debug
    "home_wpct",
    "away_wpct",
    "wpct_diff_l5",
    "wpct_diff_ytd",
] if c in out.columns]

output_df = out[final_cols].copy()
display(output_df)


,home_team,home_ytd_home_record,away_team,away_ytd_away_record,home_ytd_home_wpct,away_ytd_away_wpct,model_proba,l5_ha_proba,ytd_ha_proba,signal,model_moneyline,home_wpct,away_wpct,wpct_diff_ytd
0,MIN,19-17-1,HOU,19-15-2,0.527027,0.555556,0.614,0.5797,0.526681,CORE PLAY,-159.0,0.6,0.4,-0.028529
1,GSW,19-15-0,BKN,8-29-0,0.558824,0.216216,0.597,0.5797,0.612785,LEAN,-148.0,0.4,0.2,0.342607
2,UTA,19-18-0,WAS,7-27-0,0.513514,0.205882,0.713,0.7189,0.60467,PASS,-248.0,0.8,0.0,0.307631
3,POR,17-17-1,MIL,15-21-0,0.5,0.416667,0.685,0.6725,0.552633,PASS,-217.0,0.8,0.2,0.083333
4,DET,26-9-1,ATL,18-16-1,0.736111,0.528571,0.668,0.6725,0.581449,PASS,-201.0,1.0,0.4,0.20754
5,DEN,24-10-0,DAL,12-23-0,0.705882,0.342857,0.650,0.6261,0.617522,PASS,-186.0,0.8,0.4,0.363025
6,CLE,18-18-0,MIA,17-17-1,0.5,0.5,0.631,0.6725,0.5333,PASS,-171.0,0.6,0.0,0.0
7,PHI,22-14-1,CHI,10-22-1,0.608108,0.318182,0.549,0.5333,0.600563,PASS,-122.0,0.4,0.4,0.289926
8,LAC,23-11-1,TOR,18-18-0,0.671429,0.5,0.541,0.5797,0.573071,PASS,-118.0,0.6,0.4,0.171429
9,IND,10-24-1,LAL,13-21-3,0.3,0.391892,0.448,0.4869,0.511981,PASS,123.0,0.0,0.2,-0.091892


In [ ]:
from datetime import datetime
import os
from zoneinfo import ZoneInfo

PHX = ZoneInfo("America/Phoenix")
run_date = datetime.now(PHX).strftime("%Y%m%d")

EXPORT_DIR = r"C:\Users\padre\MLB_Code_2024\outputs"
os.makedirs(EXPORT_DIR, exist_ok=True)

filename = f"nba_1h_matchups_probs_and_records_{run_date}.csv"
filepath = os.path.join(EXPORT_DIR, filename)

output_df.to_csv(filepath, index=False)
filepath


'C:\\Users\\padre\\MLB_Code_2024\\outputs\\nba_1h_matchups_probs_and_records_20260325.csv'

In [ ]:
import numpy as np
import pandas as pd

# ------------------------------------------------
# BIN EDGES FROM TEST SET (STATIC)
# ------------------------------------------------
bin_edges = [
    (0.449, 0.47, -0.092863),
    (0.47,  0.49,  0.065710),
    (0.49,  0.51,  0.064014),
    (0.51,  0.53,  0.003692),
    (0.53,  0.55,  0.010209),
    (0.55,  0.57,  0.015898),
    (0.57,  0.59,  0.007989),
    (0.59,  0.61,  0.010759),
    (0.61,  0.63,  0.057323),
    (0.63,  0.65, -0.039396),
    (0.65,  0.67, -0.088259),
    (0.67,  0.69, -0.118900),
]


def lookup_bin_edge(p):
    if pd.isna(p):
        return np.nan
    for lo, hi, edge in bin_edges:
        if lo < p <= hi:
            return edge
    return 0.0  # outside range → neutral

# ------------------------------------------------
# APPLY
# ------------------------------------------------
output_df["bin_edge"] = output_df["model_proba"].apply(lookup_bin_edge)

# Optional: edge-adjusted model probability (DO NOT bet this blindly)
output_df["model_proba_edge_adj"] = (
    output_df["model_proba"] + output_df["bin_edge"]
).clip(0, 1)

display(
    output_df[
        [
            "home_team",
            "away_team",
            "model_proba",
            "bin_edge",
            "l5_ha_proba",
            "ytd_ha_proba",
        ]
    ]
)


,home_team,away_team,model_proba,bin_edge,l5_ha_proba,ytd_ha_proba
0,MIN,HOU,0.614,0.057323,0.5797,0.526681
1,GSW,BKN,0.597,0.010759,0.5797,0.612785
2,UTA,WAS,0.713,0.000000,0.7189,0.60467
3,POR,MIL,0.685,-0.118900,0.6725,0.552633
4,DET,ATL,0.668,-0.088259,0.6725,0.581449
5,DEN,DAL,0.650,-0.039396,0.6261,0.617522
6,CLE,MIA,0.631,-0.039396,0.6725,0.5333
7,PHI,CHI,0.549,0.010209,0.5333,0.600563
8,LAC,TOR,0.541,0.010209,0.5797,0.573071
9,IND,LAL,0.448,0.000000,0.4869,0.511981
